Download own dataset from Roboflow through API

In [ ]:
!pip install roboflow

from roboflow import Roboflow
rf = Roboflow(api_key="0kEFsvb8Ih23sdaqvBEs")
project = rf.workspace("c3423667-uon-edu-au").project("boboiboy-s-cnn-project")
version = project.version(3)
dataset = version.download("folder")


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 361.2/361.2 kB 14.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 5.4/5.4 MB 108.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 58.4/58.4 kB 6.6 MB/s eta 0:00:00
  Attempting uninstall: typer
    Found existing installation: typer 0.27.2
    Uninstalling typer-0.27.2:
      Successfully uninstalled typer-0.27.2
loading Roboflow workspace...
loading Roboflow project...



Extracting Dataset Version Zip to BoBoiBoy's-CNN-Project-3 in folder:: 100%|██████████| 6227/6227 [00:00<00:00, 11066.37it/s]


Import required librareis for training

In [ ]:
import os
import copy
import numpy as np
import matplotlib.pyplot as plt
from PIL import Image

import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from torchvision import datasets, transforms, models

from sklearn.metrics import classification_report, confusion_matrix

import random
import io
from PIL import Image
from torchvision import transforms

Model Configuration

In [ ]:
# Dataset is availibale in Colab
# Expected layout:
# /contnent/...../
#   |____ train/<class_name>/*.jpg
#   |____ valid/<class_name>/*.jpg
#   |____ test/<class_name>/*.jpg
# Dataset root identifying
DATA_ROOT = "/content/BoBoiBoy's-CNN-Project-3"
print(f"Using dataset root: {DATA_ROOT}")


Using dataset root: /content/BoBoiBoy's-CNN-Project-3


Path, classes, training settings


In [ ]:
# Turn dataset root into three folders the model uses
TRAIN_DIR = os.path.join(DATA_ROOT, "train")
VALID_DIR = os.path.join(DATA_ROOT, "valid")
TEST_DIR = os.path.join(DATA_ROOT, "test")

# Only accepts two class name
EXPECTED_CLASSES = ["apple", "banana"]

NUM_CLASSES = None
# Number of images processed in one step
BATCH_SIZE = 32
# Number of passess over the training set
NUM_EPOCHS = 30
# Larger learning rate value means bigger weight updates
LR = 2e-4
# Small penalty on large weights, help model generalise
WEIGHT_DECAY = 1e-4
# Locating the best weights and plots are saved.
BEST_CKPT = "best_model.pth"
CONFUSION_MATRIX_PATH = "confusion_matrix.png"
TRAINING_CURVES_PATH = "training_curves.png"
TEST_CONFUSION_MATRIX_PATH = "test_confusion_matrix.png"
# Scale pixel color into the range network expect
IMAGENET_MEAN = [0.485, 0.456, 0.406]
IMAGENET_STD = [0.229, 0.224, 0.225]

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device: {device}")
print(f"Dataset root: {DATA_ROOT}")

Device: cuda
Dataset root: /content/BoBoiBoy's-CNN-Project-3


Image Augmentation using torchvision.transforms

In [ ]:
train_transform = transforms.Compose([
    transforms.RandomHorizontalFlip(),
    transforms.RandomVerticalFlip(),
    transforms.RandomRotation(15),
    transforms.RandomResizedCrop(224, scale = (0.8, 1.0)),
    transforms.ToTensor(),
    transforms.Normalize(IMAGENET_MEAN, IMAGENET_STD),
])

val_transform = transforms.Compose(
    [
        transforms.Resize(256),
        transforms.CenterCrop(224),
        transforms.ToTensor(),
        transforms.Normalize(IMAGENET_MEAN, IMAGENET_STD),
    ]
)

class ImageFolderWithTrainMapping(Dataset):

  def __init__(self, root:str, classes:list, class_to_idx:dict, transform:None):
    self.root = root
    self.classes = classes
    self.class_to_idx = class_to_idx
    self.transform = transform
    self.samples = []
    exts =(".jpg", ".jpeg", ".png", ".webp")

    for class_name in classes:
      class_dir = os.path.join(root, class_name)
      if not os.path.isdir(class_dir):
        continue
      target = class_to_idx[class_name]
      for file_name in sorted(os.listdir(class_dir)):
        if file_name.lower().endswith(exts):
          self.samples.append((os.path.join(class_dir,file_name), target))

  def __len__(self):
    return len(self.samples)

  def __getitem__(self, index):
    path, target = self.samples[index]
    image = Image.open(path).convert("RGB")
    if self.transform is not None:
      image = self.transform(image)
    return image, target

train_dataset = datasets.ImageFolder(TRAIN_DIR, transform = train_transform)
classes = train_dataset.classes
class_to_idx = train_dataset.class_to_idx

NUM_CLASSES = len(classes)

missing_expected = sorted(set(EXPECTED_CLASSES) - set(classes))
unexpected_classes = sorted(set(classes) - set(EXPECTED_CLASSES))

if missing_expected:
  raise ValueError(f"Missing expected classes: {missing_expected}")
if unexpected_classes:
  raise ValueError(f"Unexpected classes: {unexpected_classes}")

valid_dataset = ImageFolderWithTrainMapping(VALID_DIR, classes, class_to_idx, transform = val_transform)
test_dataset = ImageFolderWithTrainMapping(TEST_DIR, classes, class_to_idx, transform = val_transform)

print(f"Train dataset size: {len(train_dataset)}")
print(f"Valid dataset size: {len(valid_dataset)}")
print(f"Test dataset size: {len(test_dataset)}")
print(f"Num of classes: {NUM_CLASSES}")
print(f"Classes: {classes}")

train_loader = DataLoader(train_dataset, batch_size = BATCH_SIZE, shuffle = True, num_workers = 2, pin_memory=(device.type == "cuda"),)
valid_loader = DataLoader(valid_dataset, batch_size = BATCH_SIZE, shuffle = False, num_workers = 2,pin_memory=(device.type == "cuda"),)
test_loader = DataLoader(test_dataset, batch_size = BATCH_SIZE, shuffle = False, num_workers = 2, pin_memory=(device.type == "cuda"),)

Train dataset size: 4980
Valid dataset size: 623
Test dataset size: 622
Num of classes: 2
Classes: ['apple', 'banana']


Custom CNN architecture:
  Conv2d looks for local patterns
  BatchNorm2d keep activations on a stable scale, help training is smoother
  ReLU keep positive values and zezos out negative one
  MaxPool2d halves height and width, making later layers see a coarser image

In [ ]:
#Small CNN built from scratch
class CustomFruitCNN(nn.Module):
  def __init__(self, num_classes, dropout_rate=0.4):
    super().__init__()

    # Input: (Batch, 3, 224, 224)
    self.features = nn.Sequential(
        # Block 1 -> (B,32,112,112)
        nn.Conv2d(3,32, kernel_size=3, stride=1, padding=1, bias=False),
        nn.BatchNorm2d(32),
        nn.ReLU(inplace=True),
        nn.MaxPool2d(kernel_size=2, stride=2),
        # Block 2 -> (B,64,56,56)
        nn.Conv2d(32,64, kernel_size=3, stride=1, padding=1, bias=False),
        nn.BatchNorm2d(64),
        nn.ReLU(inplace=True),
        nn.MaxPool2d(kernel_size=2, stride=2),
        # Block 3 -> (B,128,28,28)
        nn.Conv2d(64,128, kernel_size=3, stride=1, padding=1, bias=False),
        nn.BatchNorm2d(128),
        nn.ReLU(inplace=True),
        nn.MaxPool2d(kernel_size=2, stride=2),
        # Block 4 -> (B,256,14,14)
        nn.Conv2d(128,256, kernel_size=3, stride=1, padding=1, bias=False),
        nn.BatchNorm2d(256),
        nn.ReLU(inplace=True),
        nn.MaxPool2d(kernel_size=2, stride=2),
        # Block 5 -> (B,256,7,7)
        nn.Conv2d(256,256, kernel_size=3, stride=1, padding=1, bias=False),
        nn.BatchNorm2d(256),
        nn.ReLU(inplace=True),
        nn.MaxPool2d(kernel_size=2, stride=2),

        # Global Average Pooling -> (B, 256, 1, 1)
        # Squeezing (7x7) into one 256-number vector per image.
        nn.AdaptiveAvgPool2d((1,1))

    )

    # Classifier flattern the vector above, randomly drops some neurons, model does not memorise the training set
    # End with linear layer that outputs one score per class
    self.classifier = nn.Sequential(
        nn.Flatten(),
        nn.Dropout(dropout_rate),
        nn.Linear(256, 128),
        nn.ReLU(inplace=True),
        nn.Dropout(p=dropout_rate/2),
        nn.Linear(128, num_classes)
    )

    self._init_weights()
  # Set starting weights: Kaiming initalisation for convolutions
  def _init_weights(self):
    for m in self.modules():
      if isinstance(m, nn.Conv2d):
        nn.init.kaiming_normal_(m.weight, mode="fan_out", nonlinearity="relu")
      elif isinstance(m, nn.BatchNorm2d):
        nn.init.constant_(m.weight, 1.0)
        nn.init.constant_(m.bias, 0.0)
      elif isinstance(m, nn.Linear):
        nn.init.normal_(m.weight, 0.0, 0.01)
        nn.init.constant_(m.bias, 0.0)

  def forward(self, x):
    x = self.features(x)
    return self.classifier(x)


Training Setup - Optimiser AdamW, Scheduler CosineAnnealingLR

In [ ]:
train_targets = torch.tensor([t for _, t in train_dataset.samples], dtype = torch.long)
counts = torch.bincount(train_targets, minlength = len(classes)).float()
total_sample = counts.sum()
num_classes = len(classes)
class_weights = total_sample / (num_classes * counts.clamp(min=1.0))
class_weights = class_weights.to(device)


model = CustomFruitCNN(num_classes=NUM_CLASSES, dropout_rate=0.4).to(device)

criterion = nn.CrossEntropyLoss(weight = class_weights)
optimizer = torch.optim.AdamW(model.parameters(), lr = LR, weight_decay = WEIGHT_DECAY)
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max = NUM_EPOCHS)

total_params = sum(p.numel() for p in model.parameters())
trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"Total params: {total_params:,} | Trainable: {trainable_params:,}")

Total params: 1,012,386 | Trainable: 1,012,386


Training loop and save the best checkpoint with val accuracy

In [ ]:
def run_epoch(loader, train_mode:bool):
  if train_mode:
    model.train()
  else:
    model.eval()

  loss_sum = 0.0
  correct = 0
  total = 0

  for images, targets in loader:
    images = images.to(device)
    targets = targets.to(device)

    if train_mode:
      optimizer.zero_grad()
    with torch.set_grad_enabled(train_mode):
      logits = model(images)
      loss = criterion(logits, targets)

    if train_mode:
      loss.backward()
      optimizer.step()

    loss_sum += loss.item() * images.size(0)
    preds = logits.argmax(dim = 1)
    correct += (preds == targets).sum().item()
    total += targets.size(0)

  return loss_sum / max (total,1), correct / max(total,1)



Run traiing and keep the best weights

In [ ]:
history = {
    "train_loss": [],
    "train_acc": [],
    "val_loss": [],
    "val_acc": []
}

best_val_acc = 0.0
best_state = None

for epoch in range(1, NUM_EPOCHS + 1):
  train_loss, train_acc = run_epoch(train_loader, train_mode = True)
  val_loss, val_acc = run_epoch(valid_loader, train_mode = False)
  scheduler.step()

  history["train_loss"].append(train_loss)
  history["train_acc"].append(train_acc)
  history["val_loss"].append(val_loss)
  history["val_acc"].append(val_acc)

  if val_acc > best_val_acc:
    best_val_acc = val_acc
    best_state = {
        "epoch": epoch,
        "model_state_dict": copy.deepcopy(model.state_dict()),
        "val_acc": best_val_acc,
        "classes": classes,
        "class_to_idx": class_to_idx,
    }
    torch.save(best_state, BEST_CKPT)

  print(
      f"Epoch {epoch:02d}/{NUM_EPOCHS} |"
      f"Train loss = {train_loss:.4f}, train accuracy={train_acc: .4f} |"
      f"Val loss = {val_loss:.4f}, val accuracy = {val_acc:.4f} |"
      f"Best val accuracy = {best_val_acc:.4f}"
  )
print(f"Training completed. Best val acc: {best_val_acc:.4f} --> saved to {BEST_CKPT}")


Epoch 01/30 |Train loss = 0.3797, train accuracy= 0.8333 |Val loss = 0.2418, val accuracy = 0.8716 |Best val accuracy = 0.8716
Epoch 02/30 |Train loss = 0.2430, train accuracy= 0.8819 |Val loss = 0.2381, val accuracy = 0.9005 |Best val accuracy = 0.9005
Epoch 03/30 |Train loss = 0.1910, train accuracy= 0.9124 |Val loss = 0.1163, val accuracy = 0.9551 |Best val accuracy = 0.9551
Epoch 04/30 |Train loss = 0.1771, train accuracy= 0.9249 |Val loss = 0.1092, val accuracy = 0.9599 |Best val accuracy = 0.9599
Epoch 05/30 |Train loss = 0.1444, train accuracy= 0.9394 |Val loss = 0.1118, val accuracy = 0.9486 |Best val accuracy = 0.9599
Epoch 06/30 |Train loss = 0.1301, train accuracy= 0.9450 |Val loss = 0.1753, val accuracy = 0.9470 |Best val accuracy = 0.9599
Epoch 07/30 |Train loss = 0.1267, train accuracy= 0.9462 |Val loss = 0.0794, val accuracy = 0.9743 |Best val accuracy = 0.9743
Epoch 08/30 |Train loss = 0.1284, train accuracy= 0.9492 |Val loss = 0.0827, val accuracy = 0.9663 |Best val ac

Evaluation: best checkpoint, report, confusion matrix, training curves

In [ ]:
try:
  checkpoint = torch.load(
      BEST_CKPT, map_location=device, weights_only=False
  )
except TypeError:
  checkpoint = torch.load(BEST_CKPT, map_location=device)
model.load_state_dict(checkpoint["model_state_dict"])
model.eval()

Validation classification report

In [ ]:
# Classification Report
def collect_preds(loader):
  all_preds = []
  all_targets = []
  with torch.no_grad():
    for images, targets in loader:
        images = images.to(device)
        logits = model(images)
        preds = logits.argmax(dim = 1).cpu().numpy()
        all_preds.append(preds)
        all_targets.append(targets.numpy())
  if not all_preds:
    return np.array([], dtype=int), np.array([], dtype=int)

  return np.concatenate(all_preds), np.concatenate(all_targets)


y_pred_valid, y_true_valid = collect_preds(valid_loader)
labels_list = list(range(len(classes)))

labels_list = list(range(len(classes)))
print("\n Classification report (validation): \n")
print(
    classification_report(
        y_true_valid,
        y_pred_valid,
        labels = labels_list,
        target_names = classes,
        zero_division = 0,
    )
)





 Classification report (validation): 

              precision    recall  f1-score   support

       apple       0.99      0.99      0.99       296
      banana       0.99      0.99      0.99       327

    accuracy                           0.99       623
   macro avg       0.99      0.99      0.99       623
weighted avg       0.99      0.99      0.99       623



Validation confusion matrix

In [ ]:
from torch.serialization import save
# Confusion matrix
def plot_confusion_matrix(y_true, y_pred, labels_list, class_name, save_path, title):
  cm = confusion_matrix(y_true, y_pred, labels = labels_list)
  fig_cm, ax_cm = plt.subplots(figsize = (14,12))
  im_cm = ax_cm.imshow(cm, interpolation = "nearest", cmap = plt.cm.Blues)
  ax_cm.figure.colorbar(im_cm, ax = ax_cm)
  ax_cm.set(
      xticks = np.arange(cm.shape[1]),
      yticks = np.arange(cm.shape[0]),
      xticklabels = classes,
      yticklabels = classes,
      ylabel = "True label",
      xlabel = "Predicted label",
      title = "Confusion Matrix"
  )

  plt.setp(ax_cm.get_xticklabels(), rotation = 45, ha = "right", rotation_mode = "anchor")
  fig_cm.tight_layout()
  fig_cm.savefig(save_path, dpi = 300)
  plt.close(fig_cm)
  print(f"Saved confusion matrix to {save_path}")
  return cm

cm = plot_confusion_matrix(y_true_valid, y_pred_valid, labels_list, "Validation", CONFUSION_MATRIX_PATH, "Confusion Matrix")

Saved confusion matrix to confusion_matrix.png


Plot the training curves

In [ ]:
# Training Curves
ephocs_axis = np.arange(1, len(history["train_loss"]) + 1)
fig_tr, (ax1, ax2) = plt.subplots(1, 2, figsize = (12,4))
ax1.plot(ephocs_axis, history["train_loss"], label = "Train")
ax1.plot(ephocs_axis, history["val_loss"], label = "Validation")
ax1.set_xlabel("Epoch")
ax1.set_ylabel("Loss")
ax1.set_title("Training and Validation Loss")
ax1.legend()
ax1.grid(True, alpha = 0.3)

ax2.plot(ephocs_axis, history["train_acc"], label = "Train")
ax2.plot(ephocs_axis, history["val_acc"], label = "Validation")
ax2.set_xlabel("Epoch")
ax2.set_ylabel("Loss")
ax2.set_title("Training and Validation Accuracy")
ax2.legend()
ax2.grid(True, alpha = 0.3)

fig_tr.suptitle("Training curves")
fig_tr.tight_layout()
fig_tr.savefig(TRAINING_CURVES_PATH, dpi = 300)
plt.close(fig_tr)
print(f"Saved training curves to {TRAINING_CURVES_PATH}")

Saved training curves to training_curves.png


Final score on the test set

In [ ]:
checkpoint = torch.load(BEST_CKPT, map_location = device)
model.load_state_dict(checkpoint["model_state_dict"])
model.eval()

y_pred_test, y_true_test = collect_preds(test_loader)
labels_list = list(range(len(classes)))
test_acc = float((y_pred_test == y_true_test).sum()) / len(y_true_test)

print(f"Test sample: {len(y_true_test)}")
print(f"Test accuracy: {test_acc:.4f}")

print("\n Classification report (test): \n")
print(classification_report(y_true_test, y_pred_test, labels = labels_list, target_names = classes,zero_division = 0,))


plot_confusion_matrix(y_true_test, y_pred_test, labels_list, "Test", TEST_CONFUSION_MATRIX_PATH, "Confusion Matrix (Test)")

Test sample: 622
Test accuracy: 0.9775

 Classification report (test): 

              precision    recall  f1-score   support

       apple       0.97      0.99      0.98       330
      banana       0.99      0.97      0.98       292

    accuracy                           0.98       622
   macro avg       0.98      0.98      0.98       622
weighted avg       0.98      0.98      0.98       622

Saved confusion matrix to test_confusion_matrix.png


array([[326,   4],
       [ 10, 282]])

In [ ]:
def collect_energies(loader):
  model.eval()
  enerries, lables = [], []
  with torch.no_grad():
    for images, targets in loader:
      images = images.to(device)
      logits = model(images)
      energy = -torch.logsumexp(logits, dim=1)
      enerries.append(energy.cpu())
      lables.append(targets)
    return torch.cat(enerries).numpy(), torch.cat(lables).numpy()

enerries, labels = collect_energies(test_loader)

energy_threshold = float(np.percentile(enerries, 95))
print(f"Energy threshold: {energy_threshold:.4f}")

for name, idx in class_to_idx.items():
  cls_energy = enerries[labels == idx]
  print(
      f"{name:6} n ={len(cls_energy):4d} "
      f"p95={np.percentile(cls_energy, 95):.4f} max={cls_energy.max():.4f}"
  )

Energy threshold: -1.3584
apple  n = 330 p95=-1.6188 max=-0.7468
banana n = 292 p95=-1.1530 max=-0.7271


Model Inference

In [ ]:
import torch
import torch.nn as nn
from PIL import Image
from torchvision import transforms, models


# Redifine Custom CNN architecture for deployment
#Small CNN built from scratch
class CustomFruitCNN(nn.Module):
  def __init__(self, num_classes, dropout_rate=0.4):
    super().__init__()

    # Input: (Batch, 3, 224, 224)
    self.features = nn.Sequential(
        # Block 1 -> (B,32,112,112)
        nn.Conv2d(3,32, kernel_size=3, stride=1, padding=1, bias=False),
        nn.BatchNorm2d(32),
        nn.ReLU(inplace=True),
        nn.MaxPool2d(kernel_size=2, stride=2),
        # Block 2 -> (B,64,56,56)
        nn.Conv2d(32,64, kernel_size=3, stride=1, padding=1, bias=False),
        nn.BatchNorm2d(64),
        nn.ReLU(inplace=True),
        nn.MaxPool2d(kernel_size=2, stride=2),
        # Block 3 -> (B,128,28,28)
        nn.Conv2d(64,128, kernel_size=3, stride=1, padding=1, bias=False),
        nn.BatchNorm2d(128),
        nn.ReLU(inplace=True),
        nn.MaxPool2d(kernel_size=2, stride=2),
        # Block 4 -> (B,256,14,14)
        nn.Conv2d(128,256, kernel_size=3, stride=1, padding=1, bias=False),
        nn.BatchNorm2d(256),
        nn.ReLU(inplace=True),
        nn.MaxPool2d(kernel_size=2, stride=2),
        # Block 5 -> (B,256,7,7)
        nn.Conv2d(256,256, kernel_size=3, stride=1, padding=1, bias=False),
        nn.BatchNorm2d(256),
        nn.ReLU(inplace=True),
        nn.MaxPool2d(kernel_size=2, stride=2),

        # Global Average Pooling -> (B, 256, 1, 1)
        # Squeezing (7x7) into one 256-number vector per image.
        nn.AdaptiveAvgPool2d((1,1))

    )

    # Classifier flattern the vector above, randomly drops some neurons, model does not memorise the training set
    # End with linear layer that outputs one score per class
    self.classifier = nn.Sequential(
        nn.Flatten(),
        nn.Dropout(dropout_rate),
        nn.Linear(256, 128),
        nn.ReLU(inplace=True),
        nn.Dropout(p=dropout_rate/2),
        nn.Linear(128, num_classes)
    )

    self._init_weights()
  # Set starting weights: Kaiming initalisation for convolutions
  def _init_weights(self):
    for m in self.modules():
      if isinstance(m, nn.Conv2d):
        nn.init.kaiming_normal_(m.weight, mode="fan_out", nonlinearity="relu")
      elif isinstance(m, nn.BatchNorm2d):
        nn.init.constant_(m.weight, 1.0)
        nn.init.constant_(m.bias, 0.0)
      elif isinstance(m, nn.Linear):
        nn.init.normal_(m.weight, 0.0, 0.01)
        nn.init.constant_(m.bias, 0.0)

  def forward(self, x):
    x = self.features(x)
    return self.classifier(x)


# Model configuration and Initialisation
BEST_CKPT = "/content/best_model.pth"
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# Load metadata from directly from checkpoints
checkpoint = torch.load(BEST_CKPT, map_location = device, weights_only=False)

classes = checkpoint["classes"]
class_to_idx = checkpoint["class_to_idx"]
num_classes = len(classes)

try:
  checkpoint = torch.load(
      BEST_CKPT, map_location=device, weights_only=False
  )
except TypeError:
  checkpoint = torch.load(BEST_CKPT, map_location=device)


#Instantiate and load model weights
model = CustomFruitCNN(num_classes=len(classes)).to(device)
model.load_state_dict(checkpoint["model_state_dict"])
model.eval()


# Preprocessing transform
inference_transform = transforms.Compose(
    [
        transforms.Resize(256),
        transforms.CenterCrop(224),
        transforms.ToTensor(),
        transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225]),
    ]
)

# Prediction function
def predict_image(image_path: str, conf_threshold: float = 0.90, energy_threshold: float = -4.7):
  img = Image.open(image_path).convert("RGB")
  # Preprocess: adds batch dimension --> Shape: (1,3,224,224)
  tensor = inference_transform(img).unsqueeze(0).to(device)

  with torch.no_grad():
    logits = model(tensor)
    probs = torch.softmax(logits, dim=1)[0]

    energy = -torch.logsumexp(logits, dim=1).item()

    top_prob, pred_idx = torch.max(probs, dim=0)
    top_prob = top_prob.item()
    pred_idx = pred_idx.item()

  if top_prob < conf_threshold or energy > energy_threshold:
    result_label = "Unknown / Neither Apple or Banana"
  else:
    result_label = classes[pred_idx]

# Display results in console
  results = {
      "Status": result_label,
      "Top Probability": f"{top_prob * 100:.2f}% ",
      "Energy Score": f"{energy:.4f}",
      "Detailed Probabilities": {classes[i]: f"{probs[i].item() * 100:.2f}%" for i in range(num_classes)}
  }

  return results




Try one image with (softmax and energy)

In [ ]:
# Banana
predict_image("/content/images (1).webp")

{'Status': 'banana',
 'Top Probability': '99.99% ',
 'Energy Score': '-4.8321',
 'Detailed Probabilities': {'apple': '0.01%', 'banana': '99.99%'}}

In [ ]:
# Car
predict_image("/content/images.jpeg")

{'Status': 'Unknown / Neither Apple or Banana',
 'Top Probability': '97.06% ',
 'Energy Score': '-1.8158',
 'Detailed Probabilities': {'apple': '97.06%', 'banana': '2.94%'}}

In [ ]:
# Apple
predict_image("/content/images (5).jpeg")

{'Status': 'apple',
 'Top Probability': '100.00% ',
 'Energy Score': '-6.2245',
 'Detailed Probabilities': {'apple': '100.00%', 'banana': '0.00%'}}

In [ ]:
# Carrot
predict_image("/content/carrot.jpg")

{'Status': 'Unknown / Neither Apple or Banana',
 'Top Probability': '99.95% ',
 'Energy Score': '-3.8605',
 'Detailed Probabilities': {'apple': '0.05%', 'banana': '99.95%'}}